# Repartition vs Coalesce

1) Teoría: Spark puede dividir los datos en **particiones**, que son las unidades mínimas de paralelismo. Cada partición se procesa en un solo *task*, dentro de un solo **executor core**. El número y tamaño de las particiones determina directamente el rendimiento.

- Redistribuye **todos** los datos de un DataFrame en un número de particiones.

- Siempre provoca **shuffle completo** (full shuffle): Los datos datos se mezclan entre todas las particiones, usando hash partitioning (o range si se especifica).

- Puede aumentar o disminuir el número de particiones.

- Es útil cuando:

   - Se necesita aumentar el paralelismo (más particiones).

   - Se necesita redistribuir datos sesgados (skew) antes de una operación costosa.

   - Se va a escribir a disco y se requiere archivos de tamaño uniforme.

   - Se particionará por una o varias columnas para optimizar los joins o escrituras particionadas.



In [1]:
# llamado de modulos

from pyspark.sql import SparkSession
from pyspark.sql.functions import *

In [2]:
# Crear sesión de spark

spark = (SparkSession
         .builder
         .appName("Repatarticion_Otros")
         .getOrCreate())

In [3]:
# Llamado de datos

Flight_csv = spark.read.csv(
    "/content/drive/MyDrive/flight data.csv",
    header = True,
    inferSchema=True
)


In [4]:
# Repartition (10 partes)

flight_repart = Flight_csv.repartition(10)

In [5]:
flight_repart.show()

+-----------------+------------+-----------------+--------------+--------------------+--------------+--------------------+--------------------+-------------------+-------------------+--------+-----+------+--------+-------------+-------------------------------+--------------+-------------------+
|from_airport_code|from_country|dest_airport_code|  dest_country|       aircraft_type|airline_number|        airline_name|       flight_number|     departure_time|       arrival_time|duration|stops| price|currency|co2_emissions|avg_co2_emission_for_this_route|co2_percentage|          scan_date|
+-----------------+------------+-----------------+--------------+--------------------+--------------+--------------------+--------------------+-------------------+-------------------+--------+-----+------+--------+-------------+-------------------------------+--------------+-------------------+
|              SYD|   Australia|              MEL|     Australia|Boeing 737|Boeing...|            VA|  [Virgin A

In [8]:
# ¿Cuantas particiones tiene un dataframe?

Flight_csv.rdd.getNumPartitions()

2

In [9]:
flight_repart.rdd.getNumPartitions()

10

**¿Cuántas particiones usar?**

R: Una regla común es que se particione de 2 a 4 por nucleo.

Núcleos|Particiones sugeridad|
-------|---------------------|
2      |4-8                  |
4      |8-16                 |
6      |12-24                |
8      |16-32                |

Se debe considerar:

$$
Particiones = (n\cdot2,n\cdot4)
$$

- Donde:

    - $n$: Número de núcleos en el procesador.

In [10]:
# Uso de partición (exportar csv según el país)

(
    Flight_csv
    .write
    .partitionBy("from_country")
    .parquet("/content/drive/MyDrive/flights")
)


In [11]:
# Vamos a leer nuevamente

Flight_pq_p = spark.read.parquet("/content/drive/MyDrive/flights")

In [12]:
Flight_pq_p.count()

998866

In [15]:
Flight_pq_p.show()

+-----------------+-----------------+------------+--------------------+--------------+--------------------+-------------+-------------------+-------------------+--------+-----+------+--------+-------------+-------------------------------+--------------+-------------------+------------+
|from_airport_code|dest_airport_code|dest_country|       aircraft_type|airline_number|        airline_name|flight_number|     departure_time|       arrival_time|duration|stops| price|currency|co2_emissions|avg_co2_emission_for_this_route|co2_percentage|          scan_date|from_country|
+-----------------+-----------------+------------+--------------------+--------------+--------------------+-------------+-------------------+-------------------+--------+-----+------+--------+-------------+-------------------------------+--------------+-------------------+------------+
|              FRA|              ALG|     Algeria|Airbus A320|Airbu...|            TU|          [Tunisair]|  TU745|TU745|2022-05-01 12:20:0

In [13]:
# Vamos a leer una particion (El caso China)

Flight_China = spark.read.parquet(
    "/content/drive/MyDrive/flights/from_country=China"
  )

In [14]:
Flight_China.show()

+-----------------+-----------------+------------+--------------------+--------------+--------------------+--------------------+-------------------+-------------------+--------+-----+------+--------+-------------+-------------------------------+--------------+-------------------+
|from_airport_code|dest_airport_code|dest_country|       aircraft_type|airline_number|        airline_name|       flight_number|     departure_time|       arrival_time|duration|stops| price|currency|co2_emissions|avg_co2_emission_for_this_route|co2_percentage|          scan_date|
+-----------------+-----------------+------------+--------------------+--------------+--------------------+--------------------+-------------------+-------------------+--------+-----+------+--------+-------------+-------------------------------+--------------+-------------------+
|              CAN|              ALG|     Algeria|Airbus A321neo|Bo...|         multi|[China Southern| ...|CZ3531|CZ3531|CZ3531|2022-05-01 10:00:00|2022-05-0

In [16]:
Flight_China_Chile = spark.read.parquet(
    "/content/drive/MyDrive/flights/from_country=China",
    "/content/drive/MyDrive/flights/from_country=Chile"
  )

In [17]:
Flight_China_Chile.show()

+-----------------+-----------------+------------+--------------------+--------------+--------------------+--------------------+-------------------+-------------------+--------+-----+------+--------+-------------+-------------------------------+--------------+-------------------+
|from_airport_code|dest_airport_code|dest_country|       aircraft_type|airline_number|        airline_name|       flight_number|     departure_time|       arrival_time|duration|stops| price|currency|co2_emissions|avg_co2_emission_for_this_route|co2_percentage|          scan_date|
+-----------------+-----------------+------------+--------------------+--------------+--------------------+--------------------+-------------------+-------------------+--------+-----+------+--------+-------------+-------------------------------+--------------+-------------------+
|              CAN|              ALG|     Algeria|Airbus A321neo|Bo...|         multi|[China Southern| ...|CZ3531|CZ3531|CZ3531|2022-05-01 10:00:00|2022-05-0

In [18]:
Flight_China.count()

125384

In [19]:
Flight_China_Chile.count()

162409

# Coalesce

- Disminuye el número de particiones **fusionando** las particiones existentes, evitando un shuffle completo cuando es posible.

- Spark intenta combinar particiones físicamente cercanas.

- No puede aumentar el número de particiones de forma efectiva.

- Ideal para reducir particiones después de un filtrado agreviso o antes de escribir resultados finales (archivos más pequeños).



In [21]:
# Proceso para usar un Coalesce mediante un proceso (filtros)

Flight_Algeria = Flight_csv.filter(
    col("from_country") == "Algeria"
)

In [22]:
Flight_Algeria.rdd.getNumPartitions()

2

In [23]:
## Aplicando nuevamente con df particionado en 10

Flight_Algeria = flight_repart.filter(
    col("from_country") == "Algeria"
)

In [24]:
Flight_Algeria.rdd.getNumPartitions()

10

In [25]:
Flight_Algeria.count()

15737

In [26]:
## Aplicando coalesce

Flight_Coalesced = Flight_Algeria.coalesce(1)

In [28]:
Flight_Coalesced.rdd.getNumPartitions()

1

Tabla comparativa

|Característica|repartition|coalesce|
|--------------|-----------|--------|
Shuffle| Sí, completo|Mínimiza o evita el shuffle|
Aumentar particiones|Sí|No (no es efectivo)|
Disminuir particiones|Sí|Sí (uso típico)|
Balanceo de datos|Muy bueno (redistribuye)|Puede quedaer desbalanceado|
Costo computacional|Alto|Bajo|
particionar por columna|Sí|No directamente|


## Partición por peso de archivo

Cuando se desea particionar no por columna, ni por un número de particiones, si no por tamaño de alamacenamiento. Se debe aplicar una formula:

$$
\text{N° de particiones} = \frac{\text{Peso total del archivo de datos}}{text{Peso óptimo/ideal}}
$$


In [30]:
# Partición mediante pesos

Flight_csv.explain("cost")

== Optimized Logical Plan ==
Relation [from_airport_code#17,from_country#18,dest_airport_code#19,dest_country#20,aircraft_type#21,airline_number#22,airline_name#23,flight_number#24,departure_time#25,arrival_time#26,duration#27,stops#28,price#29,currency#30,co2_emissions#31,avg_co2_emission_for_this_route#32,co2_percentage#33,scan_date#34] csv, Statistics(sizeInBytes=216.1 MiB)

== Physical Plan ==
FileScan csv [from_airport_code#17,from_country#18,dest_airport_code#19,dest_country#20,aircraft_type#21,airline_number#22,airline_name#23,flight_number#24,departure_time#25,arrival_time#26,duration#27,stops#28,price#29,currency#30,co2_emissions#31,avg_co2_emission_for_this_route#32,co2_percentage#33,scan_date#34] Batched: false, DataFilters: [], Format: CSV, Location: InMemoryFileIndex(1 paths)[file:/content/drive/MyDrive/flight data.csv], PartitionFilters: [], PushedFilters: [], ReadSchema: struct<from_airport_code:string,from_country:string,dest_airport_code:string,dest_country:string,...


In [32]:
MiB = 216.1
W = 15
num_p =MiB//W
num_p

14.0

In [34]:
Flight_p_W = Flight_csv.repartition(14)

# Shuffles

Un **Shuffle** ocurre cuando Spark necesita **redistribuir datos entre particiones**, típicamente porque en una operaciones se requiere qué registros relacionados van a estar en la misma partición.

## Operadores que típicamente provocan un Shuffle:

- **groupBy**, **agg**

- **join**

- **distinct**

- **repartition**

- **orderBy** / **sort**

- **union**

## ¿Por qué es costoso?

1. Escritura a disco: cada executor escribe los datos intermedios (shuffle files) en el disco local.

2. Transferencia por red: los datos se mueven entre nodos de clúster.

3. Deserilización/serialización: Hay un overhead de CPU.

4. Garbage collection: mayor presión en memoria.

## ¿Cómo encontrar o identificar un shuffle?

Dentro del explain se debe encontrar operadores como **Exchange hashpartitioning**, **Exchange rangepartition** o **SortMergeJoin**



In [35]:
# ¿Cómo identificar los shuffles?

Flight_csv.explain(True)

== Parsed Logical Plan ==
UnresolvedDataSource format: csv, isStreaming: false, paths: 1 provided

== Analyzed Logical Plan ==
from_airport_code: string, from_country: string, dest_airport_code: string, dest_country: string, aircraft_type: string, airline_number: string, airline_name: string, flight_number: string, departure_time: timestamp, arrival_time: timestamp, duration: int, stops: int, price: double, currency: string, co2_emissions: int, avg_co2_emission_for_this_route: int, co2_percentage: string, scan_date: timestamp
Relation [from_airport_code#17,from_country#18,dest_airport_code#19,dest_country#20,aircraft_type#21,airline_number#22,airline_name#23,flight_number#24,departure_time#25,arrival_time#26,duration#27,stops#28,price#29,currency#30,co2_emissions#31,avg_co2_emission_for_this_route#32,co2_percentage#33,scan_date#34] csv

== Optimized Logical Plan ==
Relation [from_airport_code#17,from_country#18,dest_airport_code#19,dest_country#20,aircraft_type#21,airline_number#22,air

In [36]:
flight_repart.explain(True)

== Parsed Logical Plan ==
Repartition 10, true
+- Relation [from_airport_code#17,from_country#18,dest_airport_code#19,dest_country#20,aircraft_type#21,airline_number#22,airline_name#23,flight_number#24,departure_time#25,arrival_time#26,duration#27,stops#28,price#29,currency#30,co2_emissions#31,avg_co2_emission_for_this_route#32,co2_percentage#33,scan_date#34] csv

== Analyzed Logical Plan ==
from_airport_code: string, from_country: string, dest_airport_code: string, dest_country: string, aircraft_type: string, airline_number: string, airline_name: string, flight_number: string, departure_time: timestamp, arrival_time: timestamp, duration: int, stops: int, price: double, currency: string, co2_emissions: int, avg_co2_emission_for_this_route: int, co2_percentage: string, scan_date: timestamp
Repartition 10, true
+- Relation [from_airport_code#17,from_country#18,dest_airport_code#19,dest_country#20,aircraft_type#21,airline_number#22,airline_name#23,flight_number#24,departure_time#25,arriva

In [37]:
# Ejercicio de identificación de shuffle

Titanic = spark.read.csv(
    "/content/drive/MyDrive/titanic.csv",
    header = True,
    inferSchema=True
)

Titanic.show()

+-----------+--------+------+--------------------+------+----+-----+-----+----------------+-------+-----+--------+
|PassengerId|Survived|Pclass|                Name|   Sex| Age|SibSp|Parch|          Ticket|   Fare|Cabin|Embarked|
+-----------+--------+------+--------------------+------+----+-----+-----+----------------+-------+-----+--------+
|          1|       0|     3|Braund, Mr. Owen ...|  male|22.0|    1|    0|       A/5 21171|   7.25| NULL|       S|
|          2|       1|     1|Cumings, Mrs. Joh...|female|38.0|    1|    0|        PC 17599|71.2833|  C85|       C|
|          3|       1|     3|Heikkinen, Miss. ...|female|26.0|    0|    0|STON/O2. 3101282|  7.925| NULL|       S|
|          4|       1|     1|Futrelle, Mrs. Ja...|female|35.0|    1|    0|          113803|   53.1| C123|       S|
|          5|       0|     3|Allen, Mr. Willia...|  male|35.0|    0|    0|          373450|   8.05| NULL|       S|
|          6|       0|     3|    Moran, Mr. James|  male|NULL|    0|    0|      

In [43]:
Titanic.rdd.getNumPartitions()

1

In [41]:
(
    Titanic
    .groupBy("Survived")
    .agg(mean("Age"))
).show()

+--------+------------------+
|Survived|          avg(Age)|
+--------+------------------+
|       1|28.343689655172415|
|       0| 30.62617924528302|
+--------+------------------+



In [42]:
(
    Titanic
    .groupBy("Survived")
    .agg(mean("Age"))
    .explain(True)
)

== Parsed Logical Plan ==
'Aggregate ['Survived], ['Survived, unresolvedalias('avg('Age))]
+- Relation [PassengerId#704,Survived#705,Pclass#706,Name#707,Sex#708,Age#709,SibSp#710,Parch#711,Ticket#712,Fare#713,Cabin#714,Embarked#715] csv

== Analyzed Logical Plan ==
Survived: int, avg(Age): double
Aggregate [Survived#705], [Survived#705, avg(Age#709) AS avg(Age)#978]
+- Relation [PassengerId#704,Survived#705,Pclass#706,Name#707,Sex#708,Age#709,SibSp#710,Parch#711,Ticket#712,Fare#713,Cabin#714,Embarked#715] csv

== Optimized Logical Plan ==
Aggregate [Survived#705], [Survived#705, avg(Age#709) AS avg(Age)#978]
+- Project [Survived#705, Age#709]
   +- Relation [PassengerId#704,Survived#705,Pclass#706,Name#707,Sex#708,Age#709,SibSp#710,Parch#711,Ticket#712,Fare#713,Cabin#714,Embarked#715] csv

== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- HashAggregate(keys=[Survived#705], functions=[avg(Age#709)], output=[Survived#705, avg(Age)#978])
   +- Exchange hashpartitioning(Survived#7